# Reservoir Computing with Spiking Networks

**SC-NeuroCore v3.14** — Computation from dynamics, not from training.

A liquid state machine (Maass et al. 2002) uses a random recurrent
spiking network as a fixed temporal feature extractor. Only the
readout layer is trained (ridge regression). The reservoir's
dynamics project inputs into a high-dimensional spike space where
linear separation is trivial.

This notebook demonstrates:

1. **Build** a random recurrent reservoir of LIF neurons
2. **Drive** with temporal patterns (XOR of delayed inputs)
3. **Record** reservoir state (spike counts in time bins)
4. **Train** readout via ridge regression (no backprop)
5. **Compare** reservoir quality across topologies

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore import StochasticLIFNeuron
from sc_neurocore.network.population import Population
from sc_neurocore.network.projection import Projection
from sc_neurocore.network.network import Network
from sc_neurocore.network.monitor import SpikeMonitor
from sc_neurocore.network.stimulus import PoissonInput

print("SC-NeuroCore reservoir computing demo")

## 1. Temporal XOR Task

The target: $y(t) = x(t) \oplus x(t - \Delta)$, i.e. XOR of the
input with a delayed copy. This requires the reservoir to
maintain a temporal memory of past inputs — a non-trivial
computation for a linear readout.

In [ ]:
rng = np.random.default_rng(42)

N_SAMPLES = 600
DELAY = 3  # time steps delay for XOR

# Binary input signal (0 or 1 at each time step)
x_input = rng.integers(0, 2, size=N_SAMPLES).astype(float)

# Target: XOR of current input with delayed input
y_target = np.zeros(N_SAMPLES)
for t in range(DELAY, N_SAMPLES):
    y_target[t] = float(int(x_input[t]) ^ int(x_input[t - DELAY]))

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
axes[0].step(range(50), x_input[:50], where="post", linewidth=0.8)
axes[0].set_ylabel("Input x(t)")
axes[0].set_title("Temporal XOR task")
axes[1].step(range(50), y_target[:50], where="post", linewidth=0.8, color="red")
axes[1].set_ylabel(f"Target x(t) XOR x(t-{DELAY})")
axes[1].set_xlabel("Time step")
plt.tight_layout()
plt.show()

## 2. Build the Reservoir

A 200-neuron recurrent LIF network with sparse random
connectivity (10% connection probability). The recurrent
weights are scaled to maintain stable dynamics at the
edge of chaos.

In [ ]:
N_RES = 200
DT = 0.001  # 1 ms
STEP_DURATION = 0.05  # 50 ms per input step
STEPS_PER_INPUT = int(STEP_DURATION / DT)
BIN_SIZE = STEPS_PER_INPUT  # bin spikes per input step


def run_reservoir(x_seq, n_res=N_RES, w_rec=0.03, p_conn=0.1, seed=42):
    """Run input sequence through reservoir, return binned spike counts."""
    pop = Population(StochasticLIFNeuron, n=n_res, label="res")
    proj = Projection(pop, pop, weight=w_rec, probability=p_conn, seed=seed)
    mon = SpikeMonitor(pop, label="res_spk")
    drive = PoissonInput(n=n_res, rate_hz=50.0, weight=1.0, dt=DT, seed=seed + 1)
    net = Network(pop, proj, drive, mon)

    n_steps = len(x_seq)
    state_matrix = np.zeros((n_steps, n_res))

    for step_idx in range(n_steps):
        # Inject input as additional Poisson rate modulation
        drive.rate_hz = 50.0 + x_seq[step_idx] * 150.0
        net.run(duration=STEP_DURATION, dt=DT)

        # Count spikes in this time window per neuron
        for nid, times in mon.spike_trains.items():
            t_start = step_idx * STEP_DURATION
            t_end = t_start + STEP_DURATION
            count = sum(1 for t in times if t_start <= t < t_end)
            state_matrix[step_idx, nid] = count

    return state_matrix


print(f"Reservoir: {N_RES} LIF neurons, {STEP_DURATION*1000:.0f} ms per input step")
print(f"Running {N_SAMPLES} steps...")
states = run_reservoir(x_input)
print(f"State matrix shape: {states.shape}")
print(f"Mean spikes per neuron per step: {states.mean():.2f}")

## 3. Train Linear Readout

Ridge regression: $\hat{y} = X W_{\text{out}}$ where
$W_{\text{out}} = (X^T X + \alpha I)^{-1} X^T y$.

No backpropagation through the reservoir. The reservoir
weights are never modified.

In [ ]:
# Train/test split
WASHOUT = 50  # discard transient
SPLIT = 400

X_train = states[WASHOUT:SPLIT]
y_train = y_target[WASHOUT:SPLIT]
X_test = states[SPLIT:]
y_test = y_target[SPLIT:]


def ridge_regression(X, y, alpha=1.0):
    """Closed-form ridge regression."""
    n_feat = X.shape[1]
    W = np.linalg.solve(X.T @ X + alpha * np.eye(n_feat), X.T @ y)
    return W


W_out = ridge_regression(X_train, y_train, alpha=1.0)

y_pred_train = (X_train @ W_out > 0.5).astype(float)
y_pred_test = (X_test @ W_out > 0.5).astype(float)

acc_train = np.mean(y_pred_train == y_train) * 100
acc_test = np.mean(y_pred_test == y_test) * 100

print(f"Train accuracy: {acc_train:.1f}%")
print(f"Test accuracy:  {acc_test:.1f}%")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

t_range = range(len(y_test))
axes[0].step(t_range, x_input[SPLIT:SPLIT + len(y_test)],
             where="post", linewidth=0.6)
axes[0].set_ylabel("Input")
axes[0].set_title("Test set")

raw_pred = X_test @ W_out
axes[1].plot(t_range, raw_pred, linewidth=0.6, alpha=0.7, label="Readout")
axes[1].axhline(0.5, color="red", linestyle="--", alpha=0.4)
axes[1].set_ylabel("Readout")
axes[1].legend()

axes[2].step(t_range, y_test, where="post", linewidth=0.8,
             alpha=0.5, label="Target")
axes[2].step(t_range, y_pred_test, where="post", linewidth=0.8,
             alpha=0.5, label="Predicted", linestyle="--")
axes[2].set_ylabel("XOR")
axes[2].set_xlabel("Time step")
axes[2].legend()

plt.suptitle(f"Reservoir readout — test accuracy {acc_test:.1f}%")
plt.tight_layout()
plt.show()

## 4. Reservoir Dimensionality

A good reservoir projects inputs into a high-rank space.
We measure the effective dimensionality via the singular
value spectrum of the state matrix.

In [ ]:
from numpy.linalg import svd

U, S, Vt = svd(states[WASHOUT:] - states[WASHOUT:].mean(axis=0), full_matrices=False)

explained_var = S ** 2 / np.sum(S ** 2)
cumulative = np.cumsum(explained_var)

# Effective dimensionality (participation ratio)
eff_dim = np.sum(S ** 2) ** 2 / np.sum(S ** 4)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(S[:50], "o-", markersize=3)
axes[0].set_xlabel("Singular value index")
axes[0].set_ylabel("Singular value")
axes[0].set_title("Reservoir singular value spectrum")

axes[1].plot(cumulative[:50], "o-", markersize=3)
axes[1].axhline(0.95, color="red", linestyle="--", alpha=0.4, label="95%")
n_95 = np.searchsorted(cumulative, 0.95) + 1
axes[1].axvline(n_95, color="gray", linestyle=":", alpha=0.5)
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance")
axes[1].set_title(f"Effective dim = {eff_dim:.1f}, 95% in {n_95} PCs")
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

| Component | Role | Trainable |
|-----------|------|-----------|
| Input encoding | Rate modulation of Poisson drive | No |
| Reservoir | Random recurrent LIF network | No |
| Readout | Ridge regression on binned spike counts | Yes (closed-form) |

The reservoir's computation emerges from its dynamics — no
gradient-based learning required. This makes reservoir computing
attractive for neuromorphic hardware where backprop is expensive
or impossible (FPGA, memristive crossbars).

SC-NeuroCore's equation-to-Verilog pipeline can compile the
reservoir neurons directly to FPGA, with the readout implemented
as a simple weight matrix in a `sc_dense_matrix_layer`.